SQLite 存储：SqliteSaver

In [2]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SimpleState(TypedDict):
    count: int

def increment(state: SimpleState) -> dict:
    return {"count": state.get("count", 0) + 1}

# 创建图
graph = StateGraph(SimpleState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)
# 创建 SQLite checkpointer
# checkpointer = SqliteSaver.from_conn_string("checkpoints.db")
checkpointer = SqliteSaver(sqlite3.connect("checkpoints.db", check_same_thread=False))

# 或者使用内存 SQLite（用于测试）
checkpointer = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False))

app = graph.compile(checkpointer=checkpointer)

# 使用方式与 MemorySaver 相同
config = {"configurable": {"thread_id": "user-123"}}
result = app.invoke({"count": 0}, config)
print(result)

{'count': 1}
